## Checkpoint 5E — Remaining four UK-DALE Kernel SHAP windows

Copy the **single code cell** below into the existing Colab runtime used for Checkpoints 3–5D. It uses the frozen protocol and already-verified UK training representation; it executes 20 Kernel SHAP explanations across UKDALE_02–05 and writes new evidence only after all checks pass. It does **not** run LIME or change models.


In [ ]:
# CHECKPOINT 5E — Remaining FOUR UK-DALE windows: five-seed Kernel SHAP
# Frozen 4B protocol. Only background selection varies; coalition RNG fixed.
# No detector fitting, no LIME, no threshold changes, no historical overwrites.

from pathlib import Path
from itertools import combinations
import hashlib
import importlib.metadata as im
import json
import joblib
import numpy as np
import pandas as pd
import shap

required = ['ROOT', 'D5', 'FEATURES', 'COLUMNS', 'X_train',
            'selected', 'STABILITY_PROTOCOL', 'protocol_hash']
missing = [v for v in required if v not in globals()]
assert not missing, f'STOP: Missing verified prior variables: {missing}'

# Gate A: exact environment and protocol, including unchanged JSON fingerprint.
versions = {'scikit-learn':'1.6.1', 'shap':'0.54.1',
            'numpy':'2.4.6', 'pandas':'2.2.3'}
for package, version in versions.items():
    assert im.version(package) == version, f'STOP: {package} version changed'

p = STABILITY_PROTOCOL
canonical = json.dumps(p, sort_keys=True, separators=(',', ':'))
assert hashlib.sha256(canonical.encode('utf-8')).hexdigest() == protocol_hash
assert protocol_hash == '849e9d7ef15cc3b135ac74854f491ae3c2113f39c3ef527ca789cde8e829334c'
seeds = [42, 123, 456, 789, 2026]
assert p['seeds'] == seeds and p['feature_order'] == list(FEATURES)
assert len(FEATURES) == len(COLUMNS) == 23
assert COLUMNS == [f'{feature}_z' for feature in FEATURES]
shap_cfg = p['kernel_shap']
assert shap_cfg['vary'] == 'background_sampling_seed'
assert shap_cfg['training_order']['UK-DALE'] == 'original_source_row_id'
assert shap_cfg['background_size'] == 100
assert shap_cfg['coalition_rng_seed'] == 42
assert shap_cfg['nsamples'] == 300
assert shap_cfg['l1_reg'] == 'num_features(10)'
assert shap_cfg['additivity_tolerance'] == 1e-6

ROOT = Path(ROOT)
D5 = Path(D5)
protocol_file = (ROOT / 'SUPERVISOR_FINAL_EXPERIMENTS_2026' /
                 '01_protocol/CHECKPOINT4B_STABILITY_13WINDOW_849e9d7e.json')
assert protocol_file.is_file()
assert hashlib.sha256(protocol_file.read_bytes()).hexdigest() == protocol_hash

manifest = json.loads((D5 / 'manifest.json').read_text(encoding='utf-8'))
entry = manifest['model_and_background']['UK-DALE']
arch_bg = entry['shap_background_verification']
assert arch_bg['status'] == 'NUMERICALLY_CONFIRMED'
model_path = Path(entry['model_path'])
assert model_path.is_file()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

assert sha256_file(model_path) == entry['model_sha256']
model_uk5e = joblib.load(model_path)  # Trusted, previously verified user-owned artifact.
assert model_uk5e.n_features_in_ == 23
assert len(model_uk5e.estimators_) == 300
assert model_uk5e.max_samples_ == 256
assert abs(float(model_uk5e.offset_) + 0.5) < 1e-12

def uk_score_5e(matrix):
    X = np.asarray(matrix, dtype=np.float32)
    assert X.ndim == 2 and X.shape[1] == 23 and np.isfinite(X).all()
    return -model_uk5e.decision_function(X)

# Gate B: original normalized UK training array and sampling order.
assert isinstance(X_train, np.ndarray)
assert X_train.shape == (218358, 23)
assert X_train.dtype == np.float32
assert np.isfinite(X_train).all()

index42 = np.random.default_rng(42).choice(len(X_train), size=100, replace=False)
reference_bg = X_train[index42].copy()
assert hashlib.sha256(index42.astype('<i8').tobytes()).hexdigest() == (
    arch_bg['sample_positions_sha256'])
assert hashlib.sha256(reference_bg.tobytes(order='C')).hexdigest() == (
    arch_bg['background_sha256'])
assert abs(float(uk_score_5e(reference_bg).mean()) -
           float(arch_bg['expected_value'])) <= 1e-9
print('PASS: Environment, protocol, frozen UK model and 100-row historical background')

# Gate C: selected cases, target order and original scores.
selected_path = D5 / 'selected_scaled_inputs_13_windows.csv'
assert sha256_file(selected_path) == manifest['artifact_sha256'][selected_path.name]
selected_disk = pd.read_csv(selected_path, float_precision='round_trip')
assert selected_disk['window_id'].is_unique and len(selected_disk) == 13
assert selected_disk['dataset'].value_counts().to_dict() == {'UK-DALE':5, 'AMPds2':8}

case_ids = ['UKDALE_02', 'UKDALE_03', 'UKDALE_04', 'UKDALE_05']
cases = selected_disk.set_index('window_id').loc[case_ids]
assert cases['dataset'].eq('UK-DALE').all()
inputs = cases[COLUMNS].to_numpy(dtype=np.float32)
assert inputs.shape == (4,23) and np.isfinite(inputs).all()
archived_scores = cases['anomaly_score'].to_numpy(dtype=np.float64)
recomputed_scores = uk_score_5e(inputs)
score_errors = np.abs(recomputed_scores - archived_scores)
assert np.max(score_errors) <= 1e-9
print('PASS: Four distinct untouched UK-DALE windows and original anomaly scores')

# A five-background cache: changing seed is the only treatment variable.
backgrounds = {}
for seed in seeds:
    indices = np.random.default_rng(seed).choice(len(X_train), size=100, replace=False)
    bg = X_train[indices].copy()
    assert bg.shape == (100,23) and np.isfinite(bg).all()
    backgrounds[seed] = (indices, bg)
print('PASS: Five deterministic 100-row training-only backgrounds')

# Gate D: 20 controlled explanations. Preserve the NumPy RNG state afterwards.
runs, vectors, pairs, cases_summary = [], [], [], []
previous_state = np.random.get_state()
try:
    for row_number, window_id in enumerate(case_ids):
        target = inputs[row_number:row_number+1]
        true_score = float(recomputed_scores[row_number])
        top_sets = {}
        per_case_runs = []

        for seed in seeds:
            indices, bg = backgrounds[seed]
            np.random.seed(42)   # Coalition-sampling randomness held constant.
            explainer = shap.KernelExplainer(uk_score_5e, bg)
            phi = np.asarray(explainer.shap_values(
                target, nsamples=300, l1_reg='num_features(10)'),
                dtype=np.float64).reshape(-1)
            baseline = float(np.asarray(explainer.expected_value).reshape(-1)[0])
            assert phi.shape == (23,) and np.isfinite(phi).all()
            assert abs(baseline - float(uk_score_5e(bg).mean())) <= 1e-9
            residual = abs(baseline + float(phi.sum()) - true_score)
            assert residual <= 1e-6, f'STOP: {window_id}, seed={seed}: additivity'
            feature_order = np.lexsort((np.arange(23), -np.abs(phi)))
            # Do not silently pad top-five with features of zero attribution.
            assert abs(float(phi[feature_order[4]])) > 1e-12, (
                f'STOP: {window_id}, seed={seed}: top-five includes an unselected feature')
            top = feature_order[:5].tolist()
            top_sets[seed] = set(top)
            run = dict(dataset='UK-DALE', window_id=window_id, seed=seed,
                anomaly_score=true_score, expected_value=baseline,
                additivity_residual=residual,
                background_sha256=hashlib.sha256(bg.tobytes(order='C')).hexdigest(),
                positions_sha256=hashlib.sha256(indices.astype('<i8').tobytes()).hexdigest(),
                nonzero_contributions=int(np.count_nonzero(phi)),
                top5_features='|'.join(FEATURES[i] for i in top))
            runs.append(run)
            per_case_runs.append(run)
            for i, feature in enumerate(FEATURES):
                vectors.append(dict(dataset='UK-DALE', window_id=window_id,
                    seed=seed, feature_index=i, feature=feature,
                    scaled_feature_value=float(target[0,i]),
                    shap_value=float(phi[i]), abs_shap_value=float(abs(phi[i])),
                    top5=(i in top_sets[seed])))
            print(f'PASS {window_id} seed={seed}: residual={residual:.3e}; '
                  f'top5={[FEATURES[i] for i in top]}')

        case_pairs = []
        for a,b in combinations(seeds,2):
            A,B = top_sets[a], top_sets[b]
            item = dict(dataset='UK-DALE', window_id=window_id,
                seed_a=a, seed_b=b, intersection_count=len(A&B),
                union_count=len(A|B), top5_jaccard=len(A&B)/len(A|B))
            pairs.append(item)
            case_pairs.append(item)
        assert len(case_pairs) == 10
        j = np.array([v['top5_jaccard'] for v in case_pairs], dtype=np.float64)
        cases_summary.append(dict(dataset='UK-DALE', window_id=window_id,
            protocol_sha256=protocol_hash, method='Kernel SHAP',
            vary='background_sampling_seed', coalition_rng_seed=42,
            historical_attribution_identity='NOT_ESTABLISHED',
            n_runs=5, n_seed_pairs=10,
            mean_pairwise_top5_jaccard=float(j.mean()),
            min_pairwise_top5_jaccard=float(j.min()),
            max_pairwise_top5_jaccard=float(j.max()),
            max_additivity_residual=float(max(r['additivity_residual'] for r in per_case_runs))))
finally:
    np.random.set_state(previous_state)

assert len(runs) == 20
assert len(vectors) == 4*5*23 == 460
assert len(pairs) == 4*10 == 40
assert len(cases_summary) == 4
assert set(v['window_id'] for v in runs) == set(case_ids)
assert len({(v['window_id'], v['seed'], v['feature_index']) for v in vectors}) == 460

# Gate E: new evidence folder only, with safeguards against accidental overwrite.
root_output = (ROOT/'SUPERVISOR_FINAL_EXPERIMENTS_2026'/'06_explanation_evaluation')
assert root_output.is_dir()
output = root_output/'CHECKPOINT5E_UKDALE_REMAINING4_SHAP_5SEEDS_849e9d7e'

def csv_bytes(records):
    return pd.DataFrame(records).to_csv(index=False, float_format='%.17g').encode('utf-8')

batch_meta = dict(dataset='UK-DALE', window_ids=case_ids,
    protocol_sha256=protocol_hash, method='Kernel SHAP',
    treatment='background_seed_only', coalition_rng_seed=42,
    historical_attribution_identity='NOT_ESTABLISHED',
    seeds=seeds, n_windows=4, n_runs=20, n_pairwise_comparisons=40,
    n_full_feature_records=460, historical_files_unchanged=True)
payloads = {
    'ukdale_remaining4_shap_runs.csv': csv_bytes(runs),
    'ukdale_remaining4_shap_23_features_by_seed.csv': csv_bytes(vectors),
    'ukdale_remaining4_shap_pairwise_jaccard.csv': csv_bytes(pairs),
    'ukdale_remaining4_shap_window_summary.csv': csv_bytes(cases_summary),
    'ukdale_remaining4_shap_batch_manifest.json': (
        json.dumps(batch_meta, indent=2, sort_keys=True, allow_nan=False)+'\n'
    ).encode('utf-8')
}
for filename, contents in payloads.items():
    path = output/filename
    if path.exists():
        assert path.read_bytes() == contents, f'STOP: Existing evidence differs: {path}'
output.mkdir(exist_ok=True)
for filename, contents in payloads.items():
    path = output/filename
    if not path.exists():
        with path.open('xb') as stream:
            stream.write(contents)
    assert path.read_bytes() == contents

print('\n=== CHECKPOINT 5E SUMMARY ===')
print(pd.DataFrame(cases_summary)[[
    'window_id','mean_pairwise_top5_jaccard',
    'min_pairwise_top5_jaccard','max_additivity_residual']].to_string(index=False))
print('Total verified explanations:', len(runs))
print('Total archived 23-feature records:', len(vectors))
print('Evidence directory:', output)
print('\nPASS: UKDALE_REMAINING4_SHAP_FIVE_SEED_EXPERIMENT')
print('No model fitting, no LIME, no historical overwrites.')
